# Performance

Each question is answered by a fit over the [workflow](how_it_works.md)'s measurements, with the
measurements (dots) against the fit (lines, or crosses where the cases differ in more than one
way) underneath. Every grid's own result is on the [Status page](index.ipynb). The fits are redone
on every build.

In [ ]:
%config InlineBackend.figure_formats = ["svg"]
import math

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FuncFormatter, NullFormatter

import checks
from checks import fmt_gb, fmt_s

plt.rcParams.update(
    {
        "font.size": 9,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
wf = checks.workflow() or {"cases": {}, "outcomes": {}}
NO_RESULTS = "**No results yet.**"


def cases(test):
    """The cases of one workflow test that passed, e.g. test_bc."""
    return [b for name, b in wf["cases"].items() if name.startswith(test + "[")]


def info(b, key):
    return b["extra_info"][key]


def seconds(b):
    return b["stats"]["mean"]


def gb(b):
    return info(b, "peak_mb") / 1024


def fit(terms, measured):
    """Least-squares c with measured ≈ terms @ c, one row of terms per case."""
    return np.linalg.lstsq(
        np.array(terms, float), np.array(measured, float), rcond=None
    )[0]


def error(predicted, measured):
    e = np.abs(np.array(predicted) - np.array(measured)) / np.array(measured)
    return f"median {np.median(e):.0%}, worst {e.max():.0%}"


def plain_numbers(fig):
    """Axis labels 1, 10, 1K, 1M rather than 10⁰, 10¹, 10³, 10⁶."""
    label = lambda v, _: next(
        (f"{v / s:g}{u}" for s, u in ((1e9, "B"), (1e6, "M"), (1e3, "K")) if v >= s),
        f"{v:g}",
    )
    for a in fig.axes + [
        c for a in fig.axes for c in a.child_axes
    ]:  # secondary axes too
        for axis in (a.xaxis, a.yaxis):
            axis.set_major_formatter(FuncFormatter(label))
            axis.set_minor_formatter(NullFormatter())

(bathymetry)=
## How long does a grid take to build, with its bathymetry?

In [ ]:
FULL = "GEBCO_2024.nc"  # full-resolution GEBCO (15″); the other grids use it coarsened to 1/30°
full = sorted(
    (
        b
        for b in cases("test_grid")
        if info(b, "bathymetry") == FULL and info(b, "area_deg2")
    ),
    key=lambda b: info(b, "area_deg2"),
)
coarse = [
    b
    for b in cases("test_grid")
    if info(b, "bathymetry") != FULL and info(b, "area_deg2")
]
if len(full) < 3:
    checks.show(NO_RESULTS)
else:
    A = np.array([info(b, "area_deg2") for b in full])
    s, g = np.array([seconds(b) for b in full]), np.array([gb(b) for b in full])
    # time ≈ a + b·A^p: the p from 0.5 to 2 that fits best, a and b by least squares at each
    terms = lambda p: np.column_stack([np.ones_like(A), A**p])
    p = min(
        np.arange(0.5, 2, 0.01),
        key=lambda p: np.sum((terms(p) @ fit(terms(p), s) - s) ** 2 / s**2),
    )
    (ta, tb), (ma, mb) = fit(terms(p), s), fit(terms(1), g)
    time_fit, memory_fit = (lambda a: ta + tb * a**p), (lambda a: ma + mb * a)
    checks.show(
        f"**Time ≈ {ta:.0f} + {tb:.2g} · A{{sup}}`{p:.2f}` s and memory ≈ {ma:.1f} + {mb:.3f} · A GB**, A the grid's "
        f"area in square degrees, from full GEBCO at 0.1°: a 40° square takes {fmt_s(time_fit(1600))} and "
        f"{fmt_gb(memory_fit(1600))} (fit {error(time_fit(A), s)} for time, {error(memory_fit(A), g)} for memory). "
        "GEBCO coarsened to 1/30° (squares) is far cheaper, and what the other grids use."
    )
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.4), layout="constrained")
    span = np.geomspace(A.min(), A.max(), 40)
    for a, y, f, ylabel in (
        (ax[0], seconds, time_fit, "time (s)"),
        (ax[1], gb, memory_fit, "peak memory (GB)"),
    ):
        a.loglog(span, f(span), "-", color="grey", label="fit")
        a.loglog(A, [y(b) for b in full], "o", label="GEBCO 15″")
        a.loglog(
            [info(b, "area_deg2") for b in coarse],
            [y(b) for b in coarse],
            "s",
            label="GEBCO 1/30°",
        )
        a.set(xlabel="grid area (square degrees)", ylabel=ylabel)
    ax[0].legend()
    plain_numbers(fig)
    plt.show()

(obc)=
## How long do open boundaries take?

In [ ]:
bc = cases("test_bc") + cases("test_bc_sweep")


def terms(e):
    """A case's time and memory terms: S 30-day slices, k slices per worker, w forked
    workers (0 for one), P million source points, C million domain cells."""
    S, P, C = e["slices"], e["n_src"] / 1e6, e["n_cells"] / 1e6
    k, w = math.ceil(S / e["workers"]), (
        min(e["workers"], S) if e["workers"] > 1 else 0
    )
    return [1, S, S * C, k, k * P, k * C], [1, P, C, w, w * C]


if len(bc) < 6:
    checks.show(NO_RESULTS)
else:
    t = fit([terms(b["extra_info"])[0] for b in bc], [seconds(b) for b in bc])
    m = fit([terms(b["extra_info"])[1] for b in bc], [gb(b) for b in bc])
    bc_s = lambda b: float(np.dot(t, terms(b["extra_info"])[0]))
    bc_gb = lambda b: float(np.dot(m, terms(b["extra_info"])[1]))
    sweep = lambda b, **fixed: info(b, "row") == "pacific_10deg" and all(
        info(b, k) == v for k, v in fixed.items()
    )
    by_source = sorted(
        (b for b in bc if sweep(b, days=30, workers=1)), key=lambda b: info(b, "n_src")
    )
    by_cells = sorted(cases("test_bc"), key=lambda b: info(b, "n_cells"))
    by_days = sorted(
        (b for b in bc if sweep(b, source_res_deg=0.25, workers=1)),
        key=lambda b: info(b, "days"),
    )
    checks.show(
        f"**Time ≈ {t[0]:.2g} + S·({t[1]:.2g} + {t[2]:.2g} C) + k·({t[3]:.2g} + {t[4]:.2g} P + {t[5]:.2g} C) s**, "
        "S the number of 30-day slices, k slices per worker, P million source points and C million domain cells "
        f"(fit {error([bc_s(b) for b in bc], [seconds(b) for b in bc])}). On one worker it grows in a straight line "
        f"with each: {t[4]:.0f} s per million source points and {t[2] + t[5]:.0f} s per million domain cells, every "
        "30 days. The source is CrocoDash's synthetic `reference_ocean` (10 levels; GLORYS has 50), so no "
        "download is timed."
    )
    fig, ax = plt.subplots(1, 3, figsize=(12, 3.4), layout="constrained")
    for a, rows, x, xlabel in (
        (
            ax[0],
            by_source,
            lambda b: info(b, "n_src"),
            "source points (10° square, 30 days)",
        ),
        (
            ax[1],
            by_cells,
            lambda b: info(b, "n_cells"),
            "domain cells (each grid, 30 days)",
        ),
        (ax[2], by_days, lambda b: info(b, "days"), "days (10° square)"),
    ):
        xs = [x(b) for b in rows]
        a.plot(xs, [seconds(b) for b in rows], "o", label="measured")
        a.plot(xs, [bc_s(b) for b in rows], "x", color="grey", label="fit")
        a.set(
            xlabel=xlabel,
            ylabel="time (s)",
            xscale="log" if a is not ax[2] else "linear",
        )
    ax[0].legend()
    plain_numbers(fig)
    plt.show()

## Do more workers make open boundaries faster, and what do they cost in memory?

In [ ]:
by_workers = {}  # "grid, source" -> its 360-day cases, by worker count
for b in sorted(
    (b for b in bc if info(b, "days") == 360), key=lambda b: info(b, "workers")
):
    by_workers.setdefault(
        f"{info(b, 'row')}, {info(b, 'source_res_deg'):g}° source", []
    ).append(b)
if len(bc) < 6 or not by_workers:
    checks.show(NO_RESULTS)
else:
    speedups = [seconds(rows[0]) / seconds(rows[-1]) for rows in by_workers.values()]
    checks.show(
        f"**{min(speedups):.1f}–{max(speedups):.1f}× faster on 12 workers than on 1**, over 360 days (12 slices). "
        "Each worker takes whole slices, so use a worker count that divides the slices evenly. "
        f"**Memory ≈ {m[0]:.2g} + {m[1]:.2g} P + {m[2]:.2g} C + w·({m[3]:.2g} + {m[4]:.2g} C) GB**, w the number "
        f"of workers (0 for one): {m[3]:.2f} GB more per worker, and {m[2]:.1f} GB per million domain cells "
        f"(fit {error([bc_gb(b) for b in bc], [gb(b) for b in bc])})."
    )
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.4), layout="constrained")
    for label, rows in by_workers.items():
        workers = range(1, info(rows[-1], "workers") + 1)
        for a, y, f in ((ax[0], seconds, bc_s), (ax[1], gb, bc_gb)):
            (line,) = a.plot(
                workers,
                [
                    f({"extra_info": {**rows[0]["extra_info"], "workers": w}})
                    for w in workers
                ],
                "-",
                label=label,
            )
            a.plot(
                [info(b, "workers") for b in rows],
                [y(b) for b in rows],
                "o",
                color=line.get_color(),
            )
    ax[0].set(xlabel="workers", ylabel="time (s)")
    ax[1].set(xlabel="workers", ylabel="peak memory (GB)")
    ax[0].legend()
    plt.show()

(runoff)=
## How long does runoff mapping take, and how much memory?

In [ ]:
rof = cases("test_runoff") + cases("test_runoff_sweep")
cells_weights = lambda b: (
    info(b, "n_ocn") / 1e6,
    info(b, "nnsm_n_s") / 1e9,
)  # million ocean cells C, billion weights W
if len(rof) < 3:
    checks.show(NO_RESULTS)
else:
    rt = fit([[1, *cells_weights(b)] for b in rof], [seconds(b) for b in rof])
    rm = fit([[1, cells_weights(b)[1]] for b in rof], [gb(b) for b in rof])
    rof_s = lambda b: rt[0] + rt[1] * cells_weights(b)[0] + rt[2] * cells_weights(b)[1]
    rof_gb = lambda w: rm[0] + rm[1] * w / 1e9
    checks.show(
        f"**Time ≈ {rt[0]:.0f} + {rt[1]:.0f} C + {rt[2]:.0f} W s and memory ≈ {rm[0]:.0f} + {rm[1]:.0f} W GB**, "
        "C the million ocean cells and W the billion weights in the smoothed map "
        f"(fit {error([rof_s(b) for b in rof], [seconds(b) for b in rof])} for time, "
        f"{error([rof_gb(info(b, 'nnsm_n_s')) for b in rof], [gb(b) for b in rof])} for memory). The {rt[0]:.0f} s "
        f"and {rm[0]:.0f} GB floors are reading GLOFAS's global mesh, so even a 1° box needs that much."
    )
    by_cells = sorted(cases("test_runoff"), key=lambda b: info(b, "n_ocn"))
    weights = sorted(info(b, "nnsm_n_s") for b in rof)
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.4), layout="constrained")
    ax[0].semilogx(
        [info(b, "n_ocn") for b in by_cells],
        [seconds(b) for b in by_cells],
        "o",
        label="measured",
    )
    ax[0].semilogx(
        [info(b, "n_ocn") for b in by_cells],
        [rof_s(b) for b in by_cells],
        "x",
        color="grey",
        label="fit",
    )
    ax[0].set(xlabel="ocean cells (each grid, rmax 20 km)", ylabel="time (s)")
    ax[0].legend()
    ax[1].semilogx(weights, [rof_gb(w) for w in weights], "-", color="grey")
    ax[1].semilogx([info(b, "nnsm_n_s") for b in rof], [gb(b) for b in rof], "o")
    ax[1].set(
        xlabel="weights in the smoothed map (every case)", ylabel="peak memory (GB)"
    )
    plain_numbers(fig)
    plt.show()

## How does the smoothing radius grow the runoff maps, and the coupler's memory?

In [ ]:
# CESM's coupler holds the smoothed map as two route handles (liquid and ice runoff), ~167 bytes
# a weight each, measured on CrocIndoPacific_DROF_112 (4.0e8 weights, 133 GB on the mediator node).
COUPLER_GB_PER_WEIGHT = 2 * 167 / 1024**3
swept = list(dict.fromkeys(info(b, "row") for b in cases("test_runoff_sweep")))
rows = sorted(
    (b for b in rof if info(b, "row") in swept), key=lambda b: info(b, "rmax_km")
)
if len(rows) <= len(swept):
    checks.show(NO_RESULTS)
else:
    # log weights = a grid's own intercept + p·log rmax, with one p for every grid
    c = fit(
        [
            [float(info(b, "row") == d) for d in swept] + [math.log(info(b, "rmax_km"))]
            for b in rows
        ],
        [math.log(info(b, "nnsm_n_s")) for b in rows],
    )
    p, biggest = c[-1], max(rows, key=lambda b: info(b, "nnsm_n_s"))
    checks.show(
        f"**Weights grow as rmax{{sup}}`{p:.1f}`**: doubling rmax multiplies the smoothed map's weights by {2**p:.1f}, "
        f"and CESM's coupler holds about {COUPLER_GB_PER_WEIGHT * 1e9:.0f} GB per billion weights while the model runs. "
        f"`{info(biggest, 'row')}` at rmax {info(biggest, 'rmax_km')} km has {info(biggest, 'nnsm_n_s') / 1e9:.1f} billion: "
        f"~{fmt_gb(info(biggest, 'nnsm_n_s') * COUPLER_GB_PER_WEIGHT)} in the coupler."
    )
    fig, ax = plt.subplots(figsize=(5.5, 3.6), layout="constrained")
    r = np.geomspace(
        min(info(b, "rmax_km") for b in rows), max(info(b, "rmax_km") for b in rows), 40
    )
    for d, intercept in zip(swept, c[:-1]):
        (line,) = ax.loglog(r, np.exp(intercept) * r**p, "-", label=d)
        mine = [b for b in rows if info(b, "row") == d]
        ax.loglog(
            [info(b, "rmax_km") for b in mine],
            [info(b, "nnsm_n_s") for b in mine],
            "o",
            color=line.get_color(),
        )
    ax.set(
        xlabel="rmax (km)",
        ylabel="weights in the smoothed map",
        xticks=[10, 20, 40, 80],
    )
    coupler = ax.secondary_yaxis(
        "right",
        functions=(
            lambda w: w * COUPLER_GB_PER_WEIGHT,
            lambda g: g / COUPLER_GB_PER_WEIGHT,
        ),
    )
    coupler.set_ylabel("coupler memory (GB)")
    ax.legend()
    plain_numbers(fig)
    plt.show()

(mom6)=
## How fast does MOM6 run on more tasks?

In [ ]:
scaling = checks.load(checks.INTEGRATION / "mom6_scaling.json")
groups = {
    "Bahamas sweep": [scaling["reference_sweep"]],
    "four other domains": scaling["domains"],
}
points = {
    name: [p for d in domains for p in d["points"]] for name, domains in groups.items()
}
N = np.array([p["ntasks_ocn"] for ps in points.values() for p in ps])
T = np.array([p["tot_run_time_s"] for ps in points.values() for p in ps])
a, b = fit(np.column_stack([np.ones_like(N), 1 / N]), T)
checks.show(
    f"**Run time ≈ {a:.0f} + {b:.0f} / N s**, N = NTASKS_OCN, for one month of a 100×100×10 grid on Derecho "
    f"(fit {error(a + b / N, T)}): it falls as 1/N toward a {a:.0f} s floor, so past ~{b / a:.0f} tasks more "
    "barely help. Run by hand; the scripts are in the CROC monorepo's `dev/scaling_test/`."
)
fig, ax = plt.subplots(figsize=(5.5, 3.6), layout="constrained")
n = np.geomspace(N.min(), N.max(), 40)
ax.loglog(n, a + b / n, "-", color="grey", label="fit")
for name, ps in points.items():
    ax.loglog(
        [p["ntasks_ocn"] for p in ps],
        [p["tot_run_time_s"] for p in ps],
        "o",
        label=name,
    )
ax.set(xlabel="NTASKS_OCN (N)", ylabel="run time (s)")
ax.legend()
plain_numbers(fig)
plt.show()